# Generate Raw Captions

Downloads caption text from each audio captioning dataset and saves them as `../data/rawcap_*.csv`.

| Dataset | Output file |
|---|---|
| Sound-VECaps | `rawcap_sound_ve_caps.csv` |
| Auto-ACD (AudioSet) | `rawcap_auto_acd.csv` |
| Auto-ACD (VGGSound) | `rawcap_auto_acd_vggsound.csv` |
| AudioCaps Alternative 4 | `rawcap_ac_alt_4.csv` |
| WavCaps | `rawcap_wav_caps.csv` |
| AudioCaps | `rawcap_audio_caps.csv` |
| Clotho | `rawcap_clotho.csv` |

In [1]:
import warnings; warnings.simplefilter('ignore')
import logging; logging.basicConfig(level=logging.INFO)
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

INFO:numexpr.utils:Note: detected 80 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
INFO:numexpr.utils:Note: NumExpr detected 80 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.


## Sound-VECaps

In [2]:
svc = pd.read_csv('https://zenodo.org/records/12606207/files/Sound-VECaps_full.csv')
# Remove all 'Y' in the ID
svc['ytid'] = svc['id'].apply(lambda x: x[1:])

svc

,id,caption,ytid
0,Y---1_cCGK4M,"A train runs on the railroad tracks, blowing i...",---1_cCGK4M
1,Y---2_BBVHAA,"A man with a white beard and glasses, wearing ...",---2_BBVHAA
2,Y---B_v8ZoBY,The energetic crowd at the UT club cheers on t...,---B_v8ZoBY
3,Y---EDNidJUA,"A crowd is laughing and a man is speaking, fol...",---EDNidJUA
4,Y---fcVQUf3E,Water is gurgling in the background while a bo...,---fcVQUf3E
...,...,...,...
1657118,YvGScbbpz2Ts,"A woman is speaking, music is playing, followe...",vGScbbpz2Ts
1657119,YvG_2XFm2X1U,"A motorboat is speeding through the water, cre...",vG_2XFm2X1U
1657120,YvGcjEeREfXY,An adult male is speaking about music producti...,vGcjEeREfXY
1657121,YvGd5AM-wNRk,"An adult female is speaking, while two individ...",vGd5AM-wNRk


In [3]:
svc = svc[['ytid', 'caption']]
svc.to_csv('../data/rawcap_sound_ve_caps.csv', index=None)

## Auto-ACD

In [4]:
import urllib.request
import re
from collections import OrderedDict
from pathlib import Path
import json

def download_segment_csv():
    EVAL_URL = 'http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/eval_segments.csv'
    BALANCED_TRAIN_URL = 'http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/balanced_train_segments.csv'
    UNBALANCED_TRAIN_URL = 'http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/unbalanced_train_segments.csv'

    for subset_url in [EVAL_URL, BALANCED_TRAIN_URL, UNBALANCED_TRAIN_URL]:
        subset_path = '/tmp/' + Path(subset_url).name
        if Path(subset_path).is_file():
            continue
        with open(subset_path, 'w') as f:
            subset_data = urllib.request.urlopen(subset_url).read().decode()
            f.write(subset_data)
            print('Wrote', subset_path)

def make_metadata(download=False):
    # download the original metadata.
    if download:
        download_segment_csv()

    # load label maps.
    e_df = pd.read_csv('/tmp/eval_segments.csv', skiprows=2, sep=', ', engine='python')
    e_df['split'] = 'eval_segments'
    b_df = pd.read_csv('/tmp/balanced_train_segments.csv', skiprows=2, sep=', ', engine='python')
    b_df['split'] = 'balanced_train_segments'
    u_df = pd.read_csv('/tmp/unbalanced_train_segments.csv', skiprows=2, sep=', ', engine='python')
    u_df['split'] = 'unbalanced_train_segments'
    df = pd.concat([e_df, b_df, u_df])
    df = df[['# YTID', 'positive_labels', 'split']].copy()
    df.columns = ['ytid', 'label', 'split']

    on = urllib.request.urlopen('https://raw.githubusercontent.com/audioset/ontology/master/ontology.json').read().decode()
    on = json.loads(on)
    id2name = {o['id']: o['name'] for o in on}
    id2desc = {o['id']: o['description'] for o in on}

    # clean labels.
    def remove_quotations(s):
        assert s[0] == '"' and s[-1] == '"'
        return s[1:-1]
    df.label = df.label.apply(lambda s: remove_quotations(s))

    return df, id2name, id2desc

def replace_could_be_as_suggested(cap):
    '''Replace following suggestion as suggested.
    From: 'The audio caption for the given information could be: "A car engine idles consistently, indicating a car engine knocking, possibly in an engine room."'
    To: 'A car engine idles consistently, indicating a car engine knocking, possibly in an engine room.'
    '''
    if 'be: "' not in cap:
        return cap
    return re.search('be: "(.+)"', cap).group(1)
    
def autoacd_get_ytid(ytid):
    # set ytid for auto-acd data
    if ytid[0] == 'Y':
        return ytid[1:1+11]
    return ytid[:11]

def convert_to_simple_caption(comma_separated_labels):
    # make simple caption from the original AS labels
    labels = [id2name[l] for l in comma_separated_labels.split(',')]
    simple_caption = "The sound of {}, and {}.".format(", ".join(labels[:-1]),  labels[-1]) if len(labels) > 1 else f"The sound of {labels[0]}."
    return simple_caption


df, id2name, id2desc = make_metadata(download=True)

acd = pd.read_csv('https://huggingface.co/datasets/Loie/Auto-ACD/resolve/main/train.csv')
acd.caption = acd.caption.apply(replace_could_be_as_suggested)
acd['ytid'] = acd.youtube_id.apply(autoacd_get_ytid)
df['caption'] = df.label.apply(lambda s: convert_to_simple_caption(s))

# summarize
acdic = OrderedDict()
for k, v in df.reset_index()[['ytid', 'caption']].values:
    acdic[k] = v

for k, cap in acd[['ytid', 'caption']].values:
    if k in acdic:
        acdic[k] = cap

# check
for i, j in zip(list(acdic.keys()), list(df.ytid)):
    if i != j:
        print(i, j)
    if i in ['--4gqARaEJE', '--BfvyPmVMo', '--U7joUcTCo']:
        print('OK', i, j)

df['caption1'] = acdic.values()
df = df[['ytid', 'caption1']]

OK --4gqARaEJE --4gqARaEJE
OK --BfvyPmVMo --BfvyPmVMo
OK --U7joUcTCo --U7joUcTCo


In [5]:
df.to_csv('../data/rawcap_auto_acd.csv', index=None)
df[:5]

,ytid,caption1
0,--4gqARaEJE,"The sound of Domestic animals, pets, Squeak, D..."
1,--BfvyPmVMo,The sound of Hammer.
2,--U7joUcTCo,A man's laughter abruptly interrupts as someon...
3,--i-y1v8Hy8,A female voice sings while a group of people t...
4,-0BIyqJj9ZU,"A group of people burst into laughter, with a ..."


## Preparing caption embeddings for VGGSound in Auto-ACD

In [6]:
df = pd.read_csv('https://huggingface.co/datasets/Loie/VGGSound/resolve/main/vggsound.csv', header=None)
df.columns = ['ytid', 'start', 'label', 'split']
df.ytid = [f'{yid}_{s:06d}' for yid, s in df[['ytid', 'start']].values]  # zzvSVusPPgM, 30 --> zzvSVusPPgM_000030
vggs_ids = df.ytid.values
vggs_ids


array(['---g-f_I2yQ_000001', '--0PQM4-hqg_000030', '--56QUhyDQM_000185',
       ..., 'zzvCPtdNxNo_000068', 'zzvSVusPPgM_000030',
       'zzwbG0dHLhI_000150'], dtype=object)

In [7]:
aacdtrn = pd.read_csv('https://huggingface.co/datasets/Loie/Auto-ACD/resolve/main/train.csv')
# aacdtest = pd.read_csv('https://huggingface.co/datasets/Loie/Auto-ACD/resolve/main/test.csv')  skip test samples
aacdtrn.columns = [c.replace('youtube_id', 'ytid') for c in aacdtrn.columns]
aacd = pd.concat([aacdtrn])
aacdvggs = aacd[aacd.ytid.isin(vggs_ids)]
assert len(aacdvggs) == 185197
# 1917 duplicated rows to be removed
aacdvggs = aacdvggs.loc[aacdvggs[['ytid']].drop_duplicates().index]
assert len(aacdvggs) == 185197

aacdvggs_ids = aacd.ytid.values

In [8]:
vggs_no_caps = df[~df.ytid.isin(aacdvggs_ids)]
vggs_no_caps['caption'] = vggs_no_caps.label.map(lambda x: 'The sound of ' + x + '.')
aacdvggs = pd.concat([aacdvggs, vggs_no_caps[['ytid', 'caption']]])

assert all(aacdvggs.ytid.isin(vggs_ids)) and all(df.ytid.isin(aacdvggs.ytid.values))

In [9]:
aacdvggs.to_csv('../data/rawcap_auto_acd_vggsound.csv', index=None)

## Preparing Caption Embeddings for AudioCaps Alternative 4 Captions (ACalt4)

In [10]:
df = pd.read_csv('https://raw.githubusercontent.com/KeisukeImoto/ACalt4/refs/heads/main/audiocaps_alternative_4.csv')
df.columns = ['ytid', 'caption1', 'caption2', 'caption3', 'caption4']
df.to_csv('../data/rawcap_ac_alt_4.csv', index=None)
df

,ytid,caption1,caption2,caption3,caption4
0,---1_cCGK4M,A train is moving along the tracks with the rh...,"A train swiftly moving along the tracks, accom...","A train horn blaring in the distance, blending...","The unmistakable sound of a train, with the cl..."
1,---lTs1dxhU,A racing car speeding past in a virtual race,A car zooming around a track in a video game,The fast-paced sound of a car zooming along a ...,A dynamic sound of a vehicle racing on a track...
2,--0PQM4-hqg,Water flowing through a river with a gurgling ...,A waterfall cascading down with a rush of water,Gurgling water flowing through a peaceful land...,Natures symphony includes the gentle gurgling ...
3,--299m5_DdE,Excitement fills the indoor water park as chil...,The joyful sounds of children playing fill the...,Gurgling water and a waterfall fill the indoor...,The air in an indoor water park is filled with...
4,--2XRMjyizo,"Bird vocalizations, with chirps and tweets, fi...",Two police officers standing in front of a map,Birds chirping and tweeting in the background,Amidst the scene of two police officers studyi...
...,...,...,...,...,...
41780,zzlfP-snUeY,A bulldozer idling in a rural area,A bulldozer idles and its engine rumbles softl...,An idling engine of a vehicle in an outdoor se...,The engine of a parked bulldozer purrs quietly...
41781,zzm3dwoXY8Y,Birds chirping and cooing in a natural outdoor...,Birds chirping and cooing in an outdoor setting,A soft cooing sound coming from a group of bir...,The cooing of pigeons in an outdoor environment
41782,zzvWbSyZfr0,The snoring in this image is occasionally inte...,There is snoring and occasional speech coming ...,A young girl is peacefully sleeping on a bed i...,"In the background, there is a gentle snoring s..."
41783,zzwBazlj0Oc,The soft sound of pigeons cooing in a confined...,Birds cooing softly in a confined space,Pigeons cooing softly in a confined space,Pigeons cooing softly in a small room


## WavCaps

In [11]:
import pandas as pd
import json
import requests
from pathlib import Path
import io

# GitHubのURLは raw.githubusercontent.com 形式に統一する必要があります
wavcaps_json_files = [
    'https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/AudioSet_SL/as_final.json',
    'https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/BBC_Sound_Effects/bbc_final.json',
    'https://github.com/XinhaoMei/WavCaps/raw/refs/heads/master/data/json_files/FreeSound/fsd_final.json',
    'https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/SoundBible/sb_final.json'
]

folders, ids, captions = [], [], []
for url in wavcaps_json_files:
    print(f"Downloading: {url}")
    response = requests.get(url)
    js = response.json()['data']
    # with open(js_file) as f:
    #     js = json.load(f)['data']
    folder = Path(url).parent.name  # if 'AudioSet_SL' not in str(js_file) else ''
    folders.extend([folder for d in js])
    ids.extend([Path(d['id']).stem for d in js])
    captions.extend([d['caption'] for d in js])

df = pd.DataFrame({'ytid': [f+i for f, i in zip(folders, ids)], 'caption': captions})
df.to_csv('../data/rawcap_wav_caps.csv', index=None)
df[:5]

Downloading: https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/AudioSet_SL/as_final.json
Downloading: https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/BBC_Sound_Effects/bbc_final.json
Downloading: https://github.com/XinhaoMei/WavCaps/raw/refs/heads/master/data/json_files/FreeSound/fsd_final.json
Downloading: https://raw.githubusercontent.com/XinhaoMei/WavCaps/master/data/json_files/SoundBible/sb_final.json


,ytid,caption
0,AudioSet_SLYb0RFKhbpFJA,"Wind and a man speaking are heard, accompanied..."
1,AudioSet_SLYNQNTnl0zaqU,Objects are hit repeatedly while a cash regist...
2,AudioSet_SLY4PPmyY_-YrA,A hair dryer is heard with ticking.
3,AudioSet_SLYLvNUyQ3xuAQ,A dial tone is heard.
4,AudioSet_SLYXMl9lI7mKsM,"Men are singing, shouting, and music can be he..."


In [12]:
df.to_csv('../data/rawcap_wav_caps.csv', index=None)

## AudioCaps

In [13]:
df = pd.read_csv('https://raw.githubusercontent.com/cdjkim/audiocaps/refs/heads/master/dataset/train.csv')
df = df[['youtube_id', 'caption']]
df.columns = ['ytid', 'caption']

#  Append 'Y' to ytid: UDGBjjwyaqE -> YUDGBjjwyaqE
df.ytid = 'Y' + df.ytid

df.to_csv('../data/rawcap_audio_caps.csv', index=None)
df

,ytid,caption
0,Yr1nicOVtvkQ,A woman talks nearby as water pours
1,YUDGBjjwyaqE,Multiple clanging and clanking sounds
2,Y3eJ9RynJzP8,"The wind is blowing, insects are singing, and ..."
3,Y3eK62q7SnVU,The wind is blowing and rustling occurs
4,Y3eGXNIadwGk,Person is whistling
...,...,...
49833,YpOMFhtRo_gU,Background whirring punctuated with swipe and ...
49834,YpONs-WiLqjk,A person laughs with some banging and rattling
49835,YHW7zqURSqdo,People talk and laugh loudly nearby
49836,YpOXQNWFHJYU,A man speaking with light rainfall and a diffe...


## Clotho

In [14]:
df = pd.read_csv('https://zenodo.org/records/4783391/files/clotho_captions_development.csv')
df = df.sort_values('file_name')
df['ytid'] = df.file_name.apply(lambda x: Path(x).stem)
df = df[['ytid', 'caption_1', 'caption_2', 'caption_3', 'caption_4', 'caption_5']]
df.columns = ['ytid', 'caption1', 'caption2', 'caption3', 'caption4', 'caption5']

In [15]:
df.to_csv('../data/rawcap_clotho.csv', index=None)
df

,ytid,caption1,caption2,caption3,caption4,caption5
3614,Ambience Birds,A wild assortment of birds are chirping and ca...,Several different types of bird are tweeting a...,"Birds tweeting and chirping happily, engine in...",An assortment of wild birds are chirping and ...,Birds are chirping and making loud bird noises.
3402,typical neighborhood in Porto,people talking to and over each other in a bus...,"A child and a man are speaking to each other, ...",People in a busy market talk to and about each...,"The man yells, as the child and the man are sp...",A couple of people are talking back and forth.
3285,&quot;BCat Bites a Bit&quot;,The man is repeating words over and over with ...,A person with a clear voice is repeating some ...,A woman is repeating what she says at normal v...,Somebody speaks out loud and endlessly repeat...,A young person repeats his words while trying ...
1348,(Door) Porte entree,"A door is opened and closed, then after a paus...","A door opens and scrapes the floor and closes,...","A door opens and scrapes the floor and closes,...","Cork is popped, something fizzes, a door is op...",Someone opening and closing a door and people ...
2263,", iannaeneee iiaca, caienu niaoe iaao aaaiiaie...",A passenger is riding a city commuter train,A train rattles the tracks as it passes over t...,"Similar to the train running down the tracks, ...",Something moving similar to the train down the...,The train tracks are rattling when the train g...
...,...,...,...,...,...,...
1006,ww big windchimes fs 1 4 2011,Bells and chimes are activated by the wind.,Bells and chimes that are being activated by t...,Several large bells are ringing while wind chi...,Several large bells ringing and wind chimes in...,Toned percussion is being played with a soft m...
844,yapping-dog,A dog barking at a man walking by.,A dog is barking at a man walking by.,A small dog barking quickly at someone or some...,A small dog is barking quickly at someone.,Two dogs from nearby houses are barking at one...
3734,young goat bleating in a stable,A goat is bleating loudly while people are spe...,A goat bleats noisily while people in the back...,"A goat bleats loudly as time goes by, while pe...",a goat bleats loudly as time goes on while peo...,Humans chatter among themselves as a lamb is b...
632,zipper,A long zipper is tested by being repeatedly pu...,A zipper is being opened and closed quickly.,Someone is rapidly zipping and unzipping a zip...,a really long zipper is being pulled and teste...,a zipper being opened and closed at varying sp...
